In [ ]:
# %% [markdown]
# Stats Homework Notebook
# Examples: Uniform, Exponential, Binomial, Normal, CLT, MLE

# %%
import math
import numpy as np
import matplotlib.pyplot as plt

# نستخدم random generator ثابت لنتائج قابلة للإعادة
rng = np.random.default_rng(seed=42)

# دالة CDF للتوزيع الطبيعي القياسي N(0,1) بدون الحاجة لـ SciPy
def normal_cdf(x, mu=0.0, sigma=1.0):
    """
    CDF of Normal(mu, sigma^2) using math.erf
    """
    z = (x - mu) / (sigma * math.sqrt(2))
    return 0.5 * (1 + math.erf(z))


In [ ]:
# %% [markdown]
# ## Example: Uniform Distribution U(10, 20)
# Waiting time (minutes) for a train ~ Uniform(10, 20)
# - P(X >= 15)
# - Mean
# - Standard deviation

# %%
a, b = 10, 20  # حدود التوزيع

# احتمال الانتظار على الأقل 15 دقيقة: [15, 20]
p_at_least_15 = (b - 15) / (b - a)

# المتوسط والانحراف المعياري للتوزيع المنتظم
mu_uniform = (a + b) / 2
sigma_uniform = (b - a) / math.sqrt(12)

print("Uniform(10,20)")
print("P(X >= 15) =", p_at_least_15)
print("Mean       =", mu_uniform)
print("Std dev    =", sigma_uniform)


In [ ]:
# %% [markdown]
# ## Example: Exponential Distribution
# Average repair time = 0.5 hours -> Exponential with mean 0.5
# - Find P(X > 2 hours)

# %%
mean_time = 0.5            # μ = 0.5 hours
lam = 1 / mean_time        # λ = 1 / μ

t = 2.0                    # 2 hours
p_more_than_2 = math.exp(-lam * t)  # P(X > t) = e^{-λ t}

print("Exponential with mean 0.5 hours (λ = {:.1f})".format(lam))
print("P(X > 2 hours) =", p_more_than_2)


In [ ]:
# %% [markdown]
# ## Example-2: MCQ Test (Binomial)
# 3 questions, each with 4 choices (1 correct).
# Student answers randomly.

# %%
import math

n = 3
p = 0.25

def comb(n, k):
    """Simple combination function C(n,k) using factorials."""
    return math.factorial(n) // (math.factorial(k) * math.factorial(n - k))

def binom_pmf(k, n, p):
    return comb(n, k) * (p ** k) * ((1 - p) ** (n - k))

p_x_eq_2 = binom_pmf(2, n, p)
p_x_eq_3 = binom_pmf(3, n, p)

p_x_ge_2 = p_x_eq_2 + p_x_eq_3      # at least 2
p_x_le_2 = 1 - p_x_eq_3             # at most 2

expected_x = n * p
var_x = n * p * (1 - p)
std_x = math.sqrt(var_x)

print("Binomial(n=3, p=0.25)")
print("P(X = 2)       =", p_x_eq_2)
print("P(X >= 2)      =", p_x_ge_2)
print("P(X <= 2)      =", p_x_le_2)
print("E[X]           =", expected_x)
print("Std dev(X)     =", std_x)


In [ ]:
# %% [markdown]
# ## Example 7: TV Viewers (Normal Distribution)
# X ~ Normal(mu=1.5, sigma=0.2) millions
# 1) P(1.6 <= X <= 2)
# 2) P(X >= 1)
# 3) P(X > 2)

# %%
mu = 1.5      # mean in millions
sigma = 0.2   # std dev in millions

# 1) between 1.6 and 2
p_16_to_2 = normal_cdf(2.0, mu, sigma) - normal_cdf(1.6, mu, sigma)

# 2) at least 1
p_at_least_1m = 1 - normal_cdf(1.0, mu, sigma)

# 3) exceed 2
p_exceed_2m = 1 - normal_cdf(2.0, mu, sigma)

print("Normal(mu=1.5, sigma=0.2)")
print("P(1.6 <= X <= 2.0) =", p_16_to_2)
print("P(X >= 1.0)        =", p_at_least_1m)
print("P(X > 2.0)         =", p_exceed_2m)


In [ ]:
# %%
x_vals = np.linspace(0.8, 2.2, 400)
pdf_vals = (1/(sigma * math.sqrt(2*math.pi))) * np.exp(-((x_vals - mu)**2)/(2*sigma**2))

plt.figure()
plt.plot(x_vals, pdf_vals)
plt.title("Normal PDF for TV viewers (millions)")
plt.xlabel("Viewers (millions)")
plt.ylabel("Density")
plt.show()


In [ ]:
# %% [markdown]
# ## CLT Example with digits 0..9
# 1) Sample 200 digits from D={0,...,9} and plot histogram
# 2) Repeatedly sample 5 digits, compute mean, repeat 200 times
# 3) Plot histogram of the means (should look ~Normal)

# %%
D = np.arange(10)  # digits 0..9

# 1) 200 digits with replacement
sample_200 = rng.choice(D, size=200, replace=True)

plt.figure()
plt.hist(sample_200, bins=np.arange(-0.5, 10.5, 1), edgecolor='black')
plt.title("Histogram of 200 digits sampled from {0..9}")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.show()

# 2+3) means of samples of size 5, repeated 200 times
means = []
for _ in range(200):
    five_digits = rng.choice(D, size=5, replace=True)
    means.append(five_digits.mean())

means = np.array(means)

plt.figure()
plt.hist(means, bins=15, edgecolor='black', density=True)
plt.title("Histogram of sample means (n=5, 200 samples)")
plt.xlabel("Sample mean")
plt.ylabel("Density")
plt.show()

print("Mean of population (digits 0..9)   :", D.mean())
print("Mean of sample means               :", means.mean())
print("Std dev of sample means            :", means.std())


In [ ]:
# %% [markdown]
# ## MLE for Gaussian (Homework)
# Given observations X1,...,XN assumed i.i.d from N(μ, σ²),
# MLE estimates are:
#   μ_hat  = (1/N) * sum X_i
#   σ²_hat = (1/N) * sum (X_i - μ_hat)^2

# %%
def gaussian_mle(x):
    """
    Maximum Likelihood Estimation for μ and σ² of a 1D Gaussian.
    x: 1D numpy array of observations.
    returns (mu_hat, sigma2_hat)
    """
    x = np.asarray(x)
    N = x.size
    mu_hat = x.mean()
    sigma2_hat = np.sum((x - mu_hat) ** 2) / N  # divide by N (MLE)
    return mu_hat, sigma2_hat

# مثال تجريبي صغير
data = np.array([1.2, 0.9, 1.5, 1.3, 1.1])
mu_hat, sigma2_hat = gaussian_mle(data)

print("Data          :", data)
print("MLE mu        :", mu_hat)
print("MLE sigma^2   :", sigma2_hat)
print("MLE sigma     :", math.sqrt(sigma2_hat))
